# 06 — Weighted (WGKM) Kernel

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | [`03` SVM with kernel](03_svc_with_kernel.ipynb) | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | `**06**` Weighted (WGKM) kernel | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | [`09` Spectrum encoder & NB](09_spectrum_encoder_and_differential.ipynb) | [`10` Gappy encoder](10_gappy_encoder.ipynb) | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | [`12` Shuffler & chunker](12_shuffler_and_chunker.ipynb)

`WGKMKernel` applies positional weighting to L-mers via a spatial kernel (triangular, Gaussian, etc.). When the motif is near the **center** of training sequences, WGKM should outperform the uniform GKM kernel.

In [1]:
import numpy as np
from kmer.kernels import GKMKernel, WGKMKernel
from kmer.models import KernelSVM
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(42)
def random_seq(n, rng):
    return "".join(rng.choice(list("ACGT"), size=n))

SEQ_LEN = 30
MOTIF = "ACGTACGTACGT"
MOTIF_START = (SEQ_LEN - len(MOTIF)) // 2  # centered

rng = np.random.default_rng(42)
positives = []
for _ in range(60):
    s = list(random_seq(SEQ_LEN, rng))
    for i, c in enumerate(MOTIF):
        s[MOTIF_START + i] = c
    positives.append("".join(s))
negatives = [random_seq(SEQ_LEN, rng) for _ in range(60)]
seqs = positives + negatives
y = np.array([1]*60 + [0]*60)
seqs_tr, seqs_te, y_tr, y_te = train_test_split(seqs, y, test_size=0.3, stratify=y, random_state=42)
print(f"Train: {len(seqs_tr)}, Test: {len(seqs_te)}")

Train: 84, Test: 36


## Compare GKM (uniform) vs WGKM (positional weighting)

With the motif centered, WGKM's positional weighting should boost performance.

In [2]:
results = []
for name, kern in [
    ("GKM (uniform)", GKMKernel(L=10, k=6, d=3, kernel_type="truncated", use_rc=True)),
    ("WGKM (Gaussian, sigma=5)", WGKMKernel(L=10, k=6, d=3, kernel_type="truncated", use_rc=True,
                                              weight_kernel="gaussian", weight_sigma=5.0, weight_peak=1.0)),
    ("WGKM (Laplacian, gamma=0.1)", WGKMKernel(L=10, k=6, d=3, kernel_type="truncated", use_rc=True,
                                                 weight_kernel="laplacian", weight_gamma=0.1, weight_peak=1.0)),
]:
    clf = KernelSVM(kern, C=1.0)
    clf.fit(seqs_tr, y_tr)
    auc = roc_auc_score(y_te, clf.decision_function(seqs_te))
    results.append((name, auc))
    print(f"{name:35s}: AUC={auc:.3f}")

GKM (uniform)                      : AUC=1.000
WGKM (Gaussian, sigma=5)           : AUC=1.000


WGKM (Laplacian, gamma=0.1)        : AUC=1.000
